## Load Chunks From All Three Sources

In [3]:
import sys
import os

backend_path = os.path.abspath(os.path.join("..", "backend"))
sys.path.insert(0, backend_path)

from medrag.processing.storage import load_chunks

# Load one topic from each source as a starting sample
pubmed_chunks = load_chunks(source="pubmed", topic="diabetes", output_dir="../data/processed/chunks")
openfda_chunks = load_chunks(source="openfda", topic="diabetes", output_dir="../data/processed/chunks")
who_chunks = load_chunks(source="who", topic="diabetes", output_dir="../data/processed/chunks")

print(f"PubMed chunks (diabetes): {len(pubmed_chunks)}")
print(f"OpenFDA chunks (diabetes): {len(openfda_chunks)}")
print(f"WHO chunks (diabetes): {len(who_chunks)}")
print()
print("Sample PubMed chunk text:", pubmed_chunks[0].text[:200])

PubMed chunks (diabetes): 147
OpenFDA chunks (diabetes): 507
WHO chunks (diabetes): 165

Sample PubMed chunk text: Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.: First-degree relatives (FDRs) of type 1 diabetes (T1D) patients are a


## Estimate Total Embedding Cost

In [9]:
import json
from pathlib import Path
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")


def count_unique_tokens(base_dir="../data/processed/chunks"):
    seen_point_ids = set()
    total_tokens = 0
    total_unique_chunks = 0
    per_source = {}

    for source in ["pubmed", "openfda", "who"]:
        source_dir = Path(base_dir) / source
        source_tokens = 0
        source_chunks = 0
        for filepath in source_dir.glob("*.jsonl"):
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    data = json.loads(line)
                    if data["point_id"] in seen_point_ids:
                        continue  # already counted this exact chunk under another topic file
                    seen_point_ids.add(data["point_id"])
                    tokens = len(encoding.encode(data["text"]))
                    source_tokens += tokens
                    source_chunks += 1
        per_source[source] = {"tokens": source_tokens, "chunks": source_chunks}
        total_tokens += source_tokens
        total_unique_chunks += source_chunks

    return total_tokens, total_unique_chunks, per_source


total_tokens, total_unique_chunks, per_source = count_unique_tokens()

print(f"Total UNIQUE chunks: {total_unique_chunks}")
print(f"Total UNIQUE tokens: {total_tokens:,}")
print()
for source, stats in per_source.items():
    print(f"  {source}: {stats['chunks']} unique chunks, {stats['tokens']:,} tokens")

cost_per_million = 0.02
estimated_cost = (total_tokens / 1_000_000) * cost_per_million
print()
print(f"Estimated embedding cost (deduplicated): ${estimated_cost:.4f}")

Total UNIQUE chunks: 22688
Total UNIQUE tokens: 6,563,997

  pubmed: 4725 unique chunks, 1,709,671 tokens
  openfda: 13167 unique chunks, 3,407,011 tokens
  who: 4796 unique chunks, 1,447,315 tokens

Estimated embedding cost (deduplicated): $0.1313


## Test embedding a small batch of real chunks

In [10]:
from openai import OpenAI
import numpy as np

client = OpenAI()  # reads OPENAI_API_KEY from environment/.env

# Grab a small, diverse sample: some PubMed, some OpenFDA, some WHO
sample_chunks = pubmed_chunks[:5] + openfda_chunks[:5] + who_chunks[:5]

texts = [c.text for c in sample_chunks]

response = client.embeddings.create(model="text-embedding-3-small", input=texts)
embeddings = np.array([e.embedding for e in response.data])

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])
print()
print("Sample chunk text:", sample_chunks[0].text[:150])
print("First 5 values of its embedding vector:", embeddings[0][:5])

Number of embeddings: 15
Embedding dimension: 1536

Sample chunk text: Determinants and promotion strategies for type 1 diabetes screening in children: A qualitative study from a parental perspective.: First-degree relati
First 5 values of its embedding vector: [0.01131439 0.02978516 0.05441284 0.04214478 0.006073  ]


## Sanity Check — Does Embedding Similarity Actually Make Sense?

In [11]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare: PubMed diabetes chunk vs WHO diabetes chunk (expect high similarity)
pubmed_diabetes_vec = embeddings[0]   # first PubMed sample (diabetes topic)
who_diabetes_vec = embeddings[10]      # first WHO sample (diabetes topic)

# Compare: PubMed diabetes chunk vs an OpenFDA chunk (mixed relevance, depends on content)
openfda_vec = embeddings[5]

sim_pubmed_who = cosine_similarity(pubmed_diabetes_vec, who_diabetes_vec)
sim_pubmed_openfda = cosine_similarity(pubmed_diabetes_vec, openfda_vec)

print(f"PubMed (diabetes) vs WHO (diabetes) similarity: {sim_pubmed_who:.4f}")
print(f"PubMed (diabetes) vs OpenFDA (diabetes) similarity: {sim_pubmed_openfda:.4f}")
print()
print("WHO chunk text:", who_chunks[0].text[:150])
print("OpenFDA chunk text:", openfda_chunks[0].text[:150])

PubMed (diabetes) vs WHO (diabetes) similarity: 0.4550
PubMed (diabetes) vs OpenFDA (diabetes) similarity: 0.2566

WHO chunk text: Guidance on global monitoring for diabetes prevention and control: Guidance on global monitoring
for diabetes prevention and control
Framework, indica
OpenFDA chunk text: Glimepiride — Indications and Usage: 1 INDICATIONS AND USAGE Glimepiride tablets are indicated as an adjunct to diet and exercise to improve glycemic 


# Dedup-by-point_id Function

In [12]:
def deduplicate_chunks(chunks):
    """Keep only the first occurrence of each point_id."""
    seen = set()
    unique = []
    for chunk in chunks:
        if chunk.point_id not in seen:
            seen.add(chunk.point_id)
            unique.append(chunk)
    return unique


# Test on the diabetes samples we already loaded
all_diabetes_chunks = pubmed_chunks + openfda_chunks + who_chunks
unique_diabetes_chunks = deduplicate_chunks(all_diabetes_chunks)

print(f"Total chunks: {len(all_diabetes_chunks)}")
print(f"Unique chunks: {len(unique_diabetes_chunks)}")

Total chunks: 819
Unique chunks: 819


# Batching Function

In [13]:
MAX_TOKENS_PER_BATCH = 250_000
MAX_CHUNKS_PER_BATCH = 500


def build_batches(chunks):
    """Group chunks into batches respecting both a token budget and a chunk-count cap."""
    batches = []
    current_batch = []
    current_tokens = 0

    for chunk in chunks:
        chunk_tokens = len(encoding.encode(chunk.text))

        if current_batch and (
            current_tokens + chunk_tokens > MAX_TOKENS_PER_BATCH
            or len(current_batch) >= MAX_CHUNKS_PER_BATCH
        ):
            batches.append(current_batch)
            current_batch = []
            current_tokens = 0

        current_batch.append(chunk)
        current_tokens += chunk_tokens

    if current_batch:
        batches.append(current_batch)

    return batches


test_batches = build_batches(unique_diabetes_chunks)
print(f"Number of batches for {len(unique_diabetes_chunks)} chunks: {len(test_batches)}")
for i, batch in enumerate(test_batches):
    batch_tokens = sum(len(encoding.encode(c.text)) for c in batch)
    print(f"  Batch {i+1}: {len(batch)} chunks, {batch_tokens:,} tokens")

Number of batches for 819 chunks: 2
  Batch 1: 500 chunks, 139,449 tokens
  Batch 2: 319 chunks, 90,436 tokens
